In [18]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

options = Options()
options.add_argument("--headless")  # Run without opening a browser window
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

results = []
MAX_PAGES = 5  # Adjust as needed

for page in range(1, MAX_PAGES + 1):
    url = f"https://www.paulaschoice.com/ingredient-dictionary?page={page}&sort_by=name-ascending"
    print(f"Scraping page {page}...")

    driver.get(url)
    time.sleep(2)  # Let JS load

    soup = BeautifulSoup(driver.page_source, "html.parser")
    ingredient_names = soup.select(".IngredientList__Name-sc-12d8srs-10")

    if not ingredient_names:
        print("No more ingredients found. Stopping.")
        break

    for name_tag in ingredient_names:
        name = name_tag.text.strip()
        results.append({"Ingredient": name})

driver.quit()



Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...


In [20]:
# Save results
df = pd.DataFrame(results)
df.to_csv("paulaschoice_us_ingredients.csv", index=False)
print(f"\n✅ Done! Scraped {len(results)} ingredients and saved to 'paulaschoice_us_ingredients.csv'")


✅ Done! Scraped 50 ingredients and saved to 'paulaschoice_us_ingredients.csv'


In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

results = []
MAX_PAGES = 10  # Adjust this as needed to control how many pages to scrape

for page in range(1, MAX_PAGES + 1):
    url = f"https://www.paulaschoice.com/ingredient-dictionary.html?p={page}"  # US site URL
    print(f"Scraping page {page}...")
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch page {page}")
        break

    soup = BeautifulSoup(response.text, "html.parser")
    ingredient_cards = soup.select(".IngredientList__Item-sc-12d8srs-5")  # Adjust based on US site structure

    if not ingredient_cards:
        print("No more ingredients found. Stopping.")
        break

    for card in ingredient_cards:
        name_tag = card.select_one(".IngredientList__Name-sc-12d8srs-10")
        desc_tag = card.select_one(".IngredientList__Description-sc-12d8srs-11")
        rating_tag = card.select_one(".IngredientList__Rating-sc-12d8srs-12")

        name = name_tag.text.strip() if name_tag else ""
        description = desc_tag.text.strip() if desc_tag else ""
        rating = rating_tag.text.strip() if rating_tag else "Not Rated"

        results.append({
            "Ingredient": name,
            "Description": description,
            "Rating": rating
        })

    time.sleep(0.5)  # Be polite

# Save results to CSV
df = pd.DataFrame(results)
df.to_csv("paulaschoice_us_ingredients.csv", index=False)
print(f"\n✅ Done! Scraped {len(results)} ingredients and saved to 'paulaschoice_us_ingredients.csv'")


Scraping page 1...
No more ingredients found. Stopping.

✅ Done! Scraped 0 ingredients and saved to 'paulaschoice_us_ingredients.csv'
